In [1]:
import os
import random
import numpy as np
import pandas as pd
import torch
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

# 1. Фиксация воспроизводимости
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed)

set_seed(42)

# 2. Настройки
SUBMISSIONS_DIR = "submissions"
os.makedirs(SUBMISSIONS_DIR, exist_ok=True)
TIME_LIMIT = 1800 
BIAS_COEFF = 1.05 

# 3. Загрузка данных
print("Loading data...")
train = pd.read_parquet('data/train_solo_track.parquet')
test = pd.read_parquet('data/test_solo_track.parquet')

# Ограничим историю 21 днем для экономии RAM и фокуса на свежем тренде
cutoff_date = train['timestamp'].max() - pd.Timedelta(days=21)
train = train[train['timestamp'] > cutoff_date].copy()

# 4. FEATURE ENGINEERING: "Утренние якоря"
def add_anchors(df):
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['date'] = df['timestamp'].dt.date
    
    # Находим значения ровно в 10:30 для каждого дня и маршрута
    anchors = df[
        (df['timestamp'].dt.hour == 10) & 
        (df['timestamp'].dt.minute == 30)
    ][['route_id', 'date', 'target_1h', 'status_3']].rename(
        columns={'target_1h': 'anchor_target', 'status_3': 'anchor_status'}
    )
    
    # Приклеиваем эти значения ко всем часам этого же дня
    df = df.merge(anchors, on=['route_id', 'date'], how='left')
    
    # Заполняем пропуски (для ночных часов и тех, где нет 10:30)
    df['anchor_target'] = df['anchor_target'].fillna(method='ffill').fillna(0)
    df['anchor_status'] = df['anchor_status'].fillna(method='ffill').fillna(0)
    
    # Календарь
    df['is_working_day'] = df['timestamp'].dt.dayofweek.map({0:1, 1:1, 2:1, 3:1, 4:1, 5:0, 6:0}).astype(float)
    df.loc[df['timestamp'].dt.date == pd.to_datetime('2025-11-01').date(), 'is_working_day'] = 1.0
    
    return df

print("Creating morning anchors (10:30 AM data)...")
train = add_anchors(train)

# Для теста якоря берем из последней точки трейна (10:30 1 ноября)
last_train_point = train[train['timestamp'] == train['timestamp'].max()]
test['timestamp'] = pd.to_datetime(test['timestamp'])
test = test.merge(
    last_train_point[['route_id', 'anchor_target', 'anchor_status', 'is_working_day']], 
    on='route_id', how='left'
)

# 5. ПОДГОТОВКА ДАННЫХ
# Сортировка обязательна для TimeSeriesDataFrame
train = train.sort_values(['route_id', 'timestamp'])
train_ag = train.rename(columns={'route_id': 'item_id', 'target_1h': 'target'})
test_ag = test.rename(columns={'route_id': 'item_id'})

train_data = TimeSeriesDataFrame.from_data_frame(
    train_ag,
    id_column="item_id",
    timestamp_column="timestamp"
)

# Явно сортируем индекс для предотвращения UnsortedIndexError
train_data = train_data.sort_index()

# Фичи, которые известны в будущем
known_covariates_list = ['is_working_day', 'anchor_target', 'anchor_status']

# 6. СПЕЦИАЛИЗИРОВАННАЯ ВАЛИДАЦИЯ (Peak Hours Only)
print("Preparing specialized validation (11:00-14:30 only)...")
val_days = 5
val_cutoff = train['timestamp'].max() - pd.Timedelta(days=val_days)

# Вырезаем последние 5 дней
tuning_data_full = train_data.slice_by_time(val_cutoff, train['timestamp'].max())

# Оставляем в tuning_data ТОЛЬКО часы теста (11:00 - 14:30)
# Это заставит AutoGluon выбирать модели, которые лучше всего работают в пик
tuning_times = tuning_data_full.index.get_level_values('timestamp')
mask = (tuning_times.hour >= 11) & (tuning_times.hour <= 14)
tuning_data = tuning_data_full[mask]

print(f"Tuning data points: {len(tuning_data)}")

# 7. ОБУЧЕНИЕ
predictor = TimeSeriesPredictor(
    prediction_length=8,
    target="target",
    eval_metric="WAPE",
    freq="30min",
    known_covariates_names=known_covariates_list
)

print("\nStarting Specialized AutoGluon Fit...")
predictor.fit(
    train_data,
    tuning_data=tuning_data, 
    time_limit=TIME_LIMIT,
    presets="fast_training",
    random_seed=42
)

# 8. ИНФЕРЕНС
print("\nStarting Inference...")
future_covariates = TimeSeriesDataFrame.from_data_frame(
    test_ag[['item_id', 'timestamp'] + known_covariates_list],
    id_column="item_id",
    timestamp_column="timestamp"
)

predictions = predictor.predict(train_data, known_covariates=future_covariates)

# 9. ФОРМИРОВАНИЕ САБМИТА
print("Formatting submission...")
predictions = predictions.reset_index()
predictions = predictions.rename(columns={'item_id': 'route_id', 'mean': 'y_pred'})

test_orig = pd.read_parquet('data/test_solo_track.parquet')
test_orig['timestamp'] = pd.to_datetime(test_orig['timestamp'])
submission = test_orig.merge(predictions[['route_id', 'timestamp', 'y_pred']], on=['route_id', 'timestamp'], how='left')

# Сохраняем
sub_path_special = f"{SUBMISSIONS_DIR}/autogluon_special_no_k.csv"
submission[['id', 'y_pred']].to_csv(sub_path_special, index=False)

print(f"Done! Specialized submission saved to {sub_path_special}")
print(f"Best model: {predictor.model_best}")

Loading data...
Creating morning anchors (10:30 AM data)...


C:\Temp\ipykernel_26532\3284277610.py:49: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['anchor_target'] = df['anchor_target'].fillna(method='ffill').fillna(0)
C:\Temp\ipykernel_26532\3284277610.py:50: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['anchor_status'] = df['anchor_status'].fillna(method='ffill').fillna(0)
Beginning AutoGluon training... Time limit = 1800s
AutoGluon will save models to 'C:\Users\Николай\PycharmProjects\WBShipment\AutogluonModels\ag-20260401_174719'


Preparing specialized validation (11:00-14:30 only)...
Tuning data points: 40000

Starting Specialized AutoGluon Fit...


=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.10.1
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.19045
CPU Count:          20
Pytorch Version:    2.7.1+cu118
CUDA Version:       11.8
GPU Memory:         GPU 0: 6.00/6.00 GB
Total GPU Memory:   Free: 6.00 GB, Allocated: 0.00 GB, Total: 6.00 GB
GPU Count:          1
Memory Avail:       7.98 GB / 23.71 GB (33.7%)
Disk Space Avail:   41.62 GB / 476.13 GB (8.7%)
Setting presets to: fast_training

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': WAPE,
 'freq': '30min',
 'hyperparameters': 'very_light',
 'known_covariates_names': ['is_working_day', 'anchor_target', 'anchor_status'],
 'num_val_windows': 1,
 'prediction_length': 8,
 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
 'random_seed': 42,
 'refit_every_n_windows': 1,
 'refit_full': False,
 'skip_model_selection': False,
 'target': 'target',
 'time_limit': 1800,
 'v


Starting Inference...


Model not specified in predict, will default to the model with the best validation score: WeightedEnsemble


Formatting submission...
Done! Specialized submission saved to submissions/autogluon_special_no_k.csv
Best model: WeightedEnsemble
